<a href="https://colab.research.google.com/github/JulianMnZodd/Laboratorio_1_Analisis_de_Datos_Informatorio/blob/main/Lab1Navarro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dataset de salarios de jugadores de fútbol

**ETL - Extracción, Transformación y Carga**

# Extraccion:

In [13]:
from google.colab import drive
import pandas as pd

#Montar goole drive para acceder al archivo
drive.mount('/content/drive',force_remount=True)

ruta_archivo= '/content/drive/MyDrive/raw_wages.csv'

#Data Frame
try:
  df = pd.read_csv(ruta_archivo, sep=',',decimal=';')
  print('Extracción de datos exitosa')
except FileNotFoundError:
  print(f"El archivo '{ruta_archivo}' no fue encontrado.")
except Exception as e:
  print(f"Ocurrió un error al leer el archivo: {e}")
# Muestro los primeros registros para probar
df.head()




Mounted at /content/drive
Extracción de datos exitosa


,Name,Club,Division,Based,Nat,EU National,Caps,AT Apps,Position,Age,CR,Begins,Expires,Last Club,Last Trans. Fee,Salary
0,Cristiano Ronaldo - Portuguese,Al-Nassr (KSA) - Roshn Saudi League,Roshn Saudi League ...,Saudi Arabia (Saudi Pro League) ...,POR,Yes,200.0,673,ST (C),38.0,"7,629",31/12/2022,30/6/2025,,-,"€203,478,000 p/a"
1,Karim Benzema - French,Al-Ittihad - Roshn Saudi League,Roshn Saudi League ...,Saudi Arabia (Saudi Pro League) ...,FRA,Yes,97.0,555,ST (C),35.0,"8,250",1/7/2023,30/6/2025,,-,"€199,452,000 p/a"
2,Neymar - Brazilian,Al-Hilal (KSA) - Roshn Saudi League,Roshn Saudi League ...,Saudi Arabia (Saudi Pro League) ...,BRA,Yes,124.0,340,"AM (LC), ST (C)",31.0,"9,000",15/8/2023,30/6/2025,Paris Saint-Germain,€90M,"€149,589,000 p/a"
3,Ezri Konsa - English,Al-Hilal (KSA) - Roshn Saudi League,Roshn Saudi League ...,Saudi Arabia (Saudi Pro League) ...,ENG,No,0.0,245,D (C),25.0,"7,414",27/8/2023,30/6/2027,Aston Villa,€55M,"€74,157,000 p/a"
4,Kylian Mbappé - French,Paris Saint-Germain - Ligue 1 Uber Eats,Ligue 1 Uber Eats ...,France (Ligue 1) ...,FRA,Yes,70.0,221,"AM (RL), ST (C)",24.0,"9,415",22/7/2023,30/6/2028,Monaco,€186M,"€48,467,000 p/a"


# Limpieza:

In [14]:
# Inspeccionamos los datos de salario
print("Muestra de datos de salario:")
print(df['Salary'].head())

# Limpiamos datos
df_clean = df.copy()
df_clean = df_clean.drop_duplicates()
print(f"Eliminamos {df.duplicated().sum()} duplicados")

# CORREGIMOS la extracción de salarios
# Primero inspeccionamos el formato
print("\nFormato de salarios:")
print(df_clean['Salary'].unique()[:10])

# Extraemos correctamente los números (incluyendo comas para millones)
df_clean['Salary_str'] = df_clean['Salary'].astype(str)

# Eliminamos caracteres no numéricos excepto puntos y comas
df_clean['Salary_clean'] = df_clean['Salary_str'].str.replace(r'[^\d.,]', '', regex=True)

# Reemplazamos comas por puntos para decimales y eliminamos comas de miles
df_clean['Salary_clean'] = df_clean['Salary_clean'].str.replace(',', '', regex=True)

# Convertimos a numérico
df_clean['Salary_num'] = pd.to_numeric(df_clean['Salary_clean'], errors='coerce')

#Convertimos el nombre en str
try:
  df_clean['Name'] = df_clean['Name'].astype(str)
except:
  print("Error al convertir el nombre a str")



print(f"Salarios convertidos correctamente: {df_clean['Salary_num'].notna().sum()}")
print(f"Rango de salarios: €{df_clean['Salary_num'].min():,.0f} - €{df_clean['Salary_num'].max():,.0f}")

# Convertimos edad
df_clean['Age_num'] = pd.to_numeric(df_clean['Age'], errors='coerce')

# Creamos categorías CORREGIDAS
bins_salarios = [0, 1_000_000, 5_000_000, 10_000_000, float('inf')]
labels_salarios = ['<1M', '1M-5M', '5M-10M', '>10M']
df_clean['Categoria_Salario'] = pd.cut(df_clean['Salary_num'], bins=bins_salarios, labels=labels_salarios)

bins_edades = [0, 23, 28, 32, 50]
labels_edades = ['Joven', 'Medio', 'Senior', 'Veterano']
df_clean['Categoria_Edad'] = pd.cut(df_clean['Age_num'], bins=bins_edades, labels=labels_edades)



Muestra de datos de salario:
0     €203,478,000 p/a 
1     €199,452,000 p/a 
2     €149,589,000 p/a 
3     €74,157,000 p/a  
4     €48,467,000 p/a  
Name: Salary, dtype: object
Eliminamos 152 duplicados

Formato de salarios:
[' €203,478,000 p/a ' ' €199,452,000 p/a ' ' €149,589,000 p/a '
 ' €74,157,000 p/a  ' ' €48,467,000 p/a  ' ' €40,977,000 p/a  '
 ' €35,864,000 p/a  ' ' €29,918,000 p/a  ' ' €25,624,000 p/a  '
 ' €24,931,000 p/a  ']
Salarios convertidos correctamente: 40639
Rango de salarios: €180 - €203,478,000


# Carga

In [15]:
print("\nCARGA DEL DATASET FINAL")
df_final = df_clean.reset_index(drop=True)
display(df_final[['Name', 'Club', 'Position', 'Age_num', 'Salary_num', 'Categoria_Salario']].head(5))
print(f"\nDataset listo: {df_final.shape}")


CARGA DEL DATASET FINAL


,Name,Club,Position,Age_num,Salary_num,Categoria_Salario
0,Cristiano Ronaldo - Portuguese,Al-Nassr (KSA) - Roshn Saudi League,ST (C),38.0,203478000,>10M
1,Karim Benzema - French,Al-Ittihad - Roshn Saudi League,ST (C),35.0,199452000,>10M
2,Neymar - Brazilian,Al-Hilal (KSA) - Roshn Saudi League,"AM (LC), ST (C)",31.0,149589000,>10M
3,Ezri Konsa - English,Al-Hilal (KSA) - Roshn Saudi League,D (C),25.0,74157000,>10M
4,Kylian Mbappé - French,Paris Saint-Germain - Ligue 1 Uber Eats,"AM (RL), ST (C)",24.0,48467000,>10M



Dataset listo: (40639, 22)


# EDA

In [16]:

print("=" * 40)
print(f"Dimensiones: {df_final.shape}")
print(f"Memoria usada: {df_final.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

print("\nINFORMACIÓN DE TIPOS DE DATOS:")
print(df_final.info())

print("\nESTADÍSTICAS DESCRIPTIVAS - VARIABLES NUMÉRICAS")
stats = df_final[['Salary_num', 'Age_num']].describe()
stats_formatted = stats.style.format({
    'Salary_num': '€{:,.2f}',
    'Age_num': '{:.2f}'
})
display(stats_formatted)




Dimensiones: (40639, 22)
Memoria usada: 55.2 MB

INFORMACIÓN DE TIPOS DE DATOS:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40639 entries, 0 to 40638
Data columns (total 22 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   Name               40639 non-null  object  
 1   Club               40639 non-null  object  
 2   Division           40639 non-null  object  
 3   Based              40639 non-null  object  
 4   Nat                40639 non-null  object  
 5   EU National        40639 non-null  object  
 6   Caps               40639 non-null  object  
 7   AT Apps            40639 non-null  object  
 8   Position           40639 non-null  object  
 9   Age                40639 non-null  object  
 10  CR                 40639 non-null  object  
 11  Begins             40639 non-null  object  
 12  Expires            40639 non-null  object  
 13  Last Club          40639 non-null  object  
 14  Last Trans. Fee    406

,Salary_num,Age_num
count,"€40,639.00",40639.00
mean,"€320,056.08",25.21
std,"€2,009,730.23",5.32
min,€180.00,17.00
25%,"€16,750.00",21.00
50%,"€44,500.00",25.00
75%,"€156,000.00",29.00
max,"€203,478,000.00",45.00


Analisis de distribuciones:

In [17]:
print("\nDISTRIBUCIÓN DE SALARIOS POR CATEGORÍA")
print("=" * 40)
dist_salarios = pd.DataFrame({
    'Cantidad': df_final['Categoria_Salario'].value_counts(),
    'Porcentaje': (df_final['Categoria_Salario'].value_counts(normalize=True) * 100).round(1)
})
dist_salarios = dist_salarios.style.format({'Porcentaje': '{:.1f}%'})
display(dist_salarios)

print("\nDISTRIBUCIÓN DE EDADES POR CATEGORÍA")
print("=" * 40)
dist_edades = pd.DataFrame({
    'Cantidad': df_final['Categoria_Edad'].value_counts(),
    'Porcentaje': (df_final['Categoria_Edad'].value_counts(normalize=True) * 100).round(1)
})
dist_edades = dist_edades.style.format({'Porcentaje': '{:.1f}%'})
display(dist_edades)


DISTRIBUCIÓN DE SALARIOS POR CATEGORÍA


,Cantidad,Porcentaje
Categoria_Salario,,
<1M,38166,93.9%
1M-5M,2102,5.2%
5M-10M,245,0.6%
>10M,126,0.3%



DISTRIBUCIÓN DE EDADES POR CATEGORÍA


,Cantidad,Porcentaje
Categoria_Edad,,
Joven,16610,40.9%
Medio,12023,29.6%
Senior,7946,19.6%
Veterano,4060,10.0%


Interpretación:

In [18]:
print("1. ESTRUCTURA DEL DATASET:")
print(f"   • Dataset con {len(df_final)} jugadores y {df_final.shape[1]} variables")
print(f"   • {df_final['Salary_num'].isna().sum()} valores nulos en salarios")

salario_promedio = df_final['Salary_num'].mean()
print("\n2. DISTRIBUCIÓN DE SALARIOS:")
print(f"   • Salario promedio: €{salario_promedio:,.0f}")
print(f"   • {dist_salarios.data.loc['<1M', 'Porcentaje']}% en categoría baja (<€1M)")
print(f"   • Solo {dist_salarios.data.loc['>10M', 'Porcentaje']}% en categoría alta")

1. ESTRUCTURA DEL DATASET:
   • Dataset con 40639 jugadores y 22 variables
   • 0 valores nulos en salarios

2. DISTRIBUCIÓN DE SALARIOS:
   • Salario promedio: €320,056
   • 93.9% en categoría baja (<€1M)
   • Solo 0.3% en categoría alta


# Preguntas de Negocio:

Pregunta 1: ¿Qué ligas tienen los salarios promedio más altos?

In [19]:

print("¿Qué ligas tienen los salarios promedio más altos?")
print("=" * 50)

if 'Division' in df_final.columns:
    salarios_liga = df_final.groupby('Division').agg({
        'Salary_num': ['count', 'mean', 'max', 'std']
    }).round(0)

    salarios_liga.columns = ['Jugadores', 'Salario_Promedio', 'Salario_Maximo', 'Desviacion_Std']
    top_ligas = salarios_liga.nlargest(10, 'Salario_Promedio')

    print("TOP 10 LIGAS POR SALARIO PROMEDIO")
    top_ligas_formatted = top_ligas.style.format({
        'Salario_Promedio': '€{:,.0f}',
        'Salario_Maximo': '€{:,.0f}',
        'Desviacion_Std': '€{:,.0f}'
    })
    display(top_ligas_formatted)

    print("\nINTERPRETACIÓN:")
    mejor_liga = top_ligas.index[0]
    salario_mejor_liga = top_ligas.iloc[0]['Salario_Promedio']
    print(f"• {mejor_liga} lidera con €{salario_mejor_liga:,.0f} de promedio")
    print("• Alta desviación estándar indica gran variabilidad salarial interna")

¿Qué ligas tienen los salarios promedio más altos?
TOP 10 LIGAS POR SALARIO PROMEDIO


,Jugadores,Salario_Promedio,Salario_Maximo,Desviacion_Std
Division,,,,
Roshn Saudi League,238,"€6,460,630","€203,478,000","€21,701,106"
Premier League,1183,"€1,985,355","€22,733,000","€3,270,689"
LALIGA EA SPORTS,703,"€1,488,217","€24,233,000","€3,148,340"
QNB Stars League,111,"€1,460,676","€6,064,000","€1,130,742"
Bundesliga,838,"€1,245,329","€21,940,000","€2,325,795"
ADNOC Pro League,140,"€1,142,182","€7,978,000","€818,289"
Ligue 1 Uber Eats,661,"€972,030","€48,467,000","€2,638,946"
Saudi Second Division Group B,7,"€863,714","€1,896,000","€462,102"
Yelo League,71,"€858,070","€4,418,000","€731,297"



INTERPRETACIÓN:
•  Roshn Saudi League                                  lidera con €6,460,630 de promedio
• Alta desviación estándar indica gran variabilidad salarial interna


Pregunta 2: ¿Cómo se relaciona la edad con el nivel salarial?

In [20]:
print("\nPREGUNTA DE NEGOCIO 2")
print("¿Cómo se relaciona la edad con el nivel salarial?")
print("=" * 50)

salarios_edad = df_final.groupby('Categoria_Edad',observed=True).agg({
    'Salary_num': ['count', 'mean', 'median', 'max', 'min'],
    'Age_num': 'mean'
}).round(0)

salarios_edad.columns = ['Jugadores', 'Salario_Promedio', 'Salario_Mediano', 'Salario_Maximo', 'Salario_Minimo', 'Edad_Promedio']

print("SALARIOS POR GRUPO DE EDAD")
salarios_edad_formatted = salarios_edad.style.format({
    'Salario_Promedio': '€{:,.0f}',
    'Salario_Mediano': '€{:,.0f}',
    'Salario_Maximo': '€{:,.0f}',
    'Salario_Minimo': '€{:,.0f}',
    'Edad_Promedio': '{:.0f} años'
})
display(salarios_edad_formatted)

# Correlación y análisis adicional
correlacion = df_final['Age_num'].corr(df_final['Salary_num'])
print(f"Correlación edad-salario: {correlacion:.3f}")

print("\nINTERPRETACIÓN:")
grupo_max_salario = salarios_edad['Salario_Promedio'].idxmax()
print(f"• Los {grupo_max_salario} tienen salarios promedio más altos")
print(f"• Correlación débil ({correlacion:.3f}) sugiere otros factores influyen más")


PREGUNTA DE NEGOCIO 2
¿Cómo se relaciona la edad con el nivel salarial?
SALARIOS POR GRUPO DE EDAD


,Jugadores,Salario_Promedio,Salario_Mediano,Salario_Maximo,Salario_Minimo,Edad_Promedio
Categoria_Edad,,,,,,
Joven,16610,"€156,460","€30,500","€20,743,000",€180,20 años
Medio,12023,"€406,515","€65,500","€74,157,000",€300,26 años
Senior,7946,"€465,722","€66,000","€149,589,000",€300,30 años
Veterano,4060,"€448,229","€62,000","€203,478,000",€300,34 años


Correlación edad-salario: 0.070

INTERPRETACIÓN:
• Los Senior tienen salarios promedio más altos
• Correlación débil (0.070) sugiere otros factores influyen más


Pregunta 3: ¿Existe diferencia salarial entre nacionalidades?

In [21]:

if 'Nat' in df_final.columns:
    # Filtramos nacionalidades con al menos 50 jugadores para tener datos significativos
    nacionalidades_count = df_final['Nat'].value_counts()
    nacionalidades_significativas = nacionalidades_count[nacionalidades_count >= 50].index

    salarios_nacionalidad = df_final[df_final['Nat'].isin(nacionalidades_significativas)].groupby('Nat', observed=True).agg({
        'Salary_num': ['count', 'mean', 'median', 'max'],
        'Age_num': 'mean'
    }).round(0)

    salarios_nacionalidad.columns = ['Jugadores', 'Salario_Promedio', 'Salario_Mediano', 'Salario_Maximo', 'Edad_Promedio']
    top_nacionalidades = salarios_nacionalidad.nlargest(10, 'Salario_Promedio')

    print("TOP 10 NACIONALIDADES POR SALARIO PROMEDIO")
    top_nacionalidades_formatted = top_nacionalidades.style.format({
        'Salario_Promedio': '€{:,.0f}',
        'Salario_Mediano': '€{:,.0f}',
        'Salario_Maximo': '€{:,.0f}',
        'Edad_Promedio': '{:.0f} años'
    })
    display(top_nacionalidades_formatted)

    # Análisis de dispersión
    print("\nNACIONALIDADES CON MAYOR DISPERSIÓN SALARIAL")
    salarios_nacionalidad['Diferencia_Promedio_Mediano'] = salarios_nacionalidad['Salario_Promedio'] - salarios_nacionalidad['Salario_Mediano']
    top_dispersion = salarios_nacionalidad.nlargest(5, 'Diferencia_Promedio_Mediano')[['Salario_Promedio', 'Salario_Mediano', 'Diferencia_Promedio_Mediano']]

    top_dispersion_formatted = top_dispersion.style.format({
        'Salario_Promedio': '€{:,.0f}',
        'Salario_Mediano': '€{:,.0f}',
        'Diferencia_Promedio_Mediano': '€{:,.0f}'
    })
    display(top_dispersion_formatted)

    print("\nINTERPRETACIÓN:")
    mejor_pagada = top_nacionalidades.index[0]
    salario_mejor = top_nacionalidades.iloc[0]['Salario_Promedio']
    print(f"• Los jugadores {mejor_pagada} tienen los salarios promedio más altos (€{salario_mejor:,.0f})")
    print("• Alta diferencia entre promedio y mediana indica presencia de estrellas muy bien pagadas")
    print("• Algunas nacionalidades tienen gran dispersión: pocas estrellas elevan el promedio general")

TOP 10 NACIONALIDADES POR SALARIO PROMEDIO


,Jugadores,Salario_Promedio,Salario_Mediano,Salario_Maximo,Edad_Promedio
Nat,,,,,
BEL,216,"€1,174,711","€325,500","€20,619,000",25 años
KSA,138,"€1,141,478","€859,000","€5,056,000",27 años
NED,315,"€1,024,284","€260,000","€15,956,000",25 años
BRA,926,"€1,000,429","€120,000","€149,589,000",26 años
QAT,57,"€981,868","€818,000","€3,518,000",28 años
DEN,165,"€908,884","€449,000","€12,000,000",26 años
UAE,91,"€857,769","€866,000","€2,423,000",28 años
AUT,165,"€834,515","€234,000","€22,438,000",26 años
MEX,127,"€809,417","€641,000","€6,949,000",28 años



NACIONALIDADES CON MAYOR DISPERSIÓN SALARIAL


,Salario_Promedio,Salario_Mediano,Diferencia_Promedio_Mediano
Nat,,,
BRA,"€1,000,429","€120,000","€880,429"
BEL,"€1,174,711","€325,500","€849,211"
NED,"€1,024,284","€260,000","€764,284"
ARG,"€788,294","€106,000","€682,294"
CRO,"€773,488","€120,000","€653,488"



INTERPRETACIÓN:
• Los jugadores  BEL  tienen los salarios promedio más altos (€1,174,711)
• Alta diferencia entre promedio y mediana indica presencia de estrellas muy bien pagadas
• Algunas nacionalidades tienen gran dispersión: pocas estrellas elevan el promedio general
